In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Imports ---
import asyncio
import pandas as pd
from openai import OpenAI, AsyncOpenAI
from langchain_openai import ChatOpenAI
from langchain_neo4j import Neo4jGraph

from util.config_loader import load_config_api
from util.api_client import ApiClient
from llm.tool import (
    build_ontology_mapper_tool,
    build_patient_ner_tool,
    build_patient_ned_tool,
    build_general_medical_tool,
    build_patient_info_tool,
    build_patient_coverage_tool,
)


In [5]:
from IPython.display import JSON, Markdown, display
import json

def show_json(raw, mode="text", indent=2):
    """
    mode = "tree"  -> interactive JSON widget
    mode = "text"  -> indented JSON
    mode = "both"  -> tree + indented JSON
    """
    if isinstance(raw, str):
        raw = json.loads(raw)

    if mode in ("tree", "both"):
        display(JSON(raw))

    if mode in ("text", "both"):
        pretty = json.dumps(raw, indent=indent, ensure_ascii=False)
        display(Markdown(f"```json\n{pretty}\n```"))

# Connectors

In [10]:
url_emb = load_config_api("embedding", path="../config.ini")
api = ApiClient(url_emb)
body, status, headers = api.post('/v1/embeddings', {'input': ['Who are you?', 'What is your name?']})
print("Status:", status)
print("Headers:", headers)
print("Body:", body)

Status: 200
Headers: {'Content-Length': '154086', 'Content-Type': 'application/json', 'Date': 'Wed, 11 Mar 2026 11:22:32 GMT', 'Server': 'uvicorn', 'Connection': 'close'}
Body: {'id': 'embd-93e8c814c5619598', 'object': 'list', 'created': 1773228152, 'model': 'Alibaba-NLP/gte-Qwen2-7B-instruct', 'data': [{'index': 0, 'object': 'embedding', 'embedding': [0.006900744512677193, 0.0006434477982111275, 1.0563850992184598e-05, 0.018053840845823288, 0.003245214931666851, -0.00984754879027605, 0.013279270380735397, 0.0016599087975919247, -0.022082382813096046, 0.02461887337267399, 0.007534867152571678, -0.015144336968660355, 0.005072979722172022, -0.006378525868058205, 0.02327602542936802, 0.010444370098412037, 0.02148556150496006, -0.019993508234620094, -0.007049949839711189, -0.005110281053930521, 0.019993508234620094, 0.0022007781080901623, -0.007721373811364174, -0.0003706818970385939, 0.006192019674926996, -0.01618877425789833, -0.015442747622728348, -0.032675959169864655, 0.00428965222090

In [11]:
url_llm = load_config_api("llm", path="../config.ini")
chat_client = ChatOpenAI(
    api_key="EMPTY",
    base_url=url_llm,
    model_name="google/medgemma-4b-it",
    temperature=0,
    max_tokens=8192,
    top_p=0.9,
    frequency_penalty=0.2,
    presence_penalty=0.0,
)

In [12]:
graph_client = Neo4jGraph(
    url="bolt://localhost:7687",
    username="neo4j",
    password="password",
    database="nodes2026",
    enhanced_schema=True,
)

# Open LLM Tests and Virtualized Resources


## Single Request

In [13]:
client = OpenAI(
        api_key="EMPTY",                      
        base_url=url_llm
    )
resp = client.chat.completions.create(
    model="google/medgemma-4b-it",
    messages=[
        {"role": "user", "content": "Who are you?"}
    ],
    temperature=0,
)
response = resp.choices[0].message.content
response

'I am Gemma, an open-weights AI assistant. I am a large language model trained by Google DeepMind.\n'

## Batch Request


In [14]:
prompts = [
    "Explain the difference between type 1 and type 2 diabetes in simple terms. In one sentence.",
    "List the main symptoms of pneumonia and when a patient should see a doctor. In one sentence.",
    "What lifestyle changes can help reduce high blood pressure? In one sentence.",
    "Describe the importance of vaccination in preventing infectious diseases. In one sentence.",
    "What are the common treatments for seasonal allergies? In one sentence.",
    "How does regular exercise benefit mental health? In one sentence.",
    "What dietary changes can help manage cholesterol levels? In one sentence.",
]

async def run_batch(prompts, concurrency=8, max_tokens=128):
    client = AsyncOpenAI(api_key="EMPTY", base_url=url_llm)
    sem = asyncio.Semaphore(concurrency)

    async def one(p):
        async with sem:
            r = await client.chat.completions.create(
                model="google/medgemma-4b-it",
                messages=[{"role": "user", "content": p}],
                temperature=0.0,
                max_tokens=max_tokens,
                stream=False,
            )
            return r.choices[0].message.content

    tasks = [asyncio.create_task(one(p)) for p in prompts]
    out = await asyncio.gather(*tasks)
    await client.close()
    return out

answers = await run_batch(prompts, concurrency=8)
for q, a in zip(prompts, answers):
    print(q, "→", a)


Explain the difference between type 1 and type 2 diabetes in simple terms. In one sentence. → Type 1 diabetes is an autoimmune disease where the body attacks and destroys insulin-producing cells, while type 2 diabetes is a condition where the body doesn't use insulin properly, often due to resistance and/or insufficient production.

List the main symptoms of pneumonia and when a patient should see a doctor. In one sentence. → Pneumonia symptoms include cough (often with phlegm), fever, chills, shortness of breath, and chest pain. See a doctor if you have these symptoms, especially if you are elderly, have a chronic condition, or are immunocompromised.

What lifestyle changes can help reduce high blood pressure? In one sentence. → Lifestyle changes like diet modification (reducing sodium and increasing potassium), regular exercise, weight management, limiting alcohol consumption, and quitting smoking can significantly reduce high blood pressure.

Describe the importance of vaccination i

## Virtualized Resources

### Materialize Data

In [15]:
graph_client.query("MATCH (n:HpoPhenotype) RETURN n.label OFFSET 100 LIMIT 10 ")

[{'n.label': 'Abnormal bladder morphology'},
 {'n.label': 'Impaired continence'},
 {'n.label': 'Dilatation of the bladder'},
 {'n.label': 'Abnormality of the male genitalia'},
 {'n.label': 'Hernia of the abdominal wall'},
 {'n.label': 'Increased inflammatory response'},
 {'n.label': 'Abnormal prostate morphology'},
 {'n.label': 'Functional abnormality of male internal genitalia'},
 {'n.label': 'Abnormal male reproductive system physiology'},
 {'n.label': 'Hypogonadism'}]

### Virtualized Data

In [17]:
pd.read_csv("../data/sample/patient_annotated.csv").head(5)

,PatientId,EncounterID,Encounter.period.start,Encounter.class,Encounter.reasonCode,ChiefComplaint,Condition,Comorbidities,Observation[vitals],Observation[key],...,MedicationStatement,Encounter.diagnosis.rank,Encounter.hospitalization.dischargeDisposition,Plan/FollowUp,CourseTrend,Notes,Narrative,NER_Entities,NED_Entities,ICD10_Codes
0,P003,E001,2017-06-02,Ambulatory,Progressive leg tingling and burning,Leg numbness and electric-like pains,Multiple mononeuropathy,NaN,"137/81 mmHg, HR 78 bpm, RR 16, Temp 37.1°C, Sp...",Neurologic exam: patchy distal sensory loss an...,...,Gabapentin 100 mg PO BID,1,Home,Neurology follow-up in 4 weeks,Worsened,Progressive sensory symptoms,The patient reported several weeks of burning ...,"[{'source': 'concat', 'start': 14, 'end': 24, ...","[{'source': 'concat', 'start': 14, 'end': 24, ...","['G57.1', 'T28.1', 'S84', 'R52', 'G58.7', 'S94..."
1,P003,E002,2018-03-15,Emergency,Severe headache with nausea,Persistent pressure-like headache and nausea,"Aseptic meningitis, fungal etiology suspected",Multiple mononeuropathy,"139/85 mmHg, HR 92 bpm, RR 18, Temp 38.0°C, Sp...","Neuro exam: mild neck stiffness, no focal defi...",...,Ondansetron 4 mg IV PRN,1,Observation,Monitor neurologic status and repeat CSF if sy...,Improved,Headache and nausea partially relieved,The patient presented to the emergency departm...,"[{'source': 'concat', 'start': 14, 'end': 26, ...","[{'source': 'concat', 'start': 14, 'end': 26, ...","['R51', 'R11', 'G03.0', 'B48.8', 'R51', 'F40.0..."
2,P003,E003,2019-01-29,Ambulatory,Jaw pain and stress-related behaviors,Teeth grinding and morning jaw soreness,Bruxism,"Multiple mononeuropathy, prior aseptic meningitis","132/80 mmHg, HR 76 bpm, RR 16, Temp 36.8°C, Sp...",Oral exam: dental wear patterns and tender mas...,...,Hydroxyzine 25 mg PO PRN for anxiety,3,Home,Dental guard fitting and referral to behaviora...,Stable,Stress-related nocturnal grinding,The patient reported several months of waking ...,"[{'source': 'concat', 'start': 0, 'end': 4, 't...","[{'source': 'concat', 'start': 0, 'end': 4, 't...","['S03.4', 'F43.8', 'K07.6', 'K07.6', 'S03.4', ..."
3,P003,E004,2019-10-12,Ambulatory,Eye pain and blurred vision,"Blurred vision, floaters, and eye discomfort",Posterior uveitis,"Multiple mononeuropathy, bruxism, prior asepti...","128/79 mmHg, HR 74 bpm, RR 17, Temp 36.7°C, Sp...",Ophthalmic exam: vitreous cells and mild retin...,...,Prednisolone acetate 1% ophthalmic drops QID t...,2,Home,Follow-up with ophthalmology in 1 week,Improved,Posterior segment inflammation partially respo...,The patient presented with several days of blu...,"[{'source': 'concat', 'start': 14, 'end': 24, ...","[{'source': 'concat', 'start': 14, 'end': 24, ...","['H57.1', 'H53.9', 'H43.0', 'H57.1', 'H43.0', ..."
4,P003,E005,2020-08-03,Emergency,Palpitations and dizziness,Sudden pounding heartbeat with lightheadedness,Paroxysmal tachycardia,"Multiple mononeuropathy, posterior uveitis","142/88 mmHg, HR 148 bpm, RR 19, Temp 36.9°C, S...","ECG: narrow-complex tachycardia, spontaneously...",...,Metoprolol 25 mg PO daily,1,Home,Cardiology follow-up within 2 weeks,Improved,Rhythm stabilized after spontaneous conversion,"While walking up a short flight of stairs, the...","[{'source': 'concat', 'start': 14, 'end': 26, ...","[{'source': 'concat', 'start': 14, 'end': 26, ...","['R00.2', 'R00.2', 'I47', 'R00.2', 'I47']"


In [19]:
graph_client.query("MATCH (n:Encounter) RETURN n.label")

Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `Encounter` does not exist in database `nodes2026`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=10, offset=9>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 1, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (n:Encounter) RETURN n.label'


[]

In [22]:
graph_client.query("""
    CALL apoc.dv.query('encounter', {patientId:'P003'}) YIELD node AS v
    RETURN v LIMIT 1
""")

[{'v': {'Condition': 'Multiple mononeuropathy',
   'Encounter.period.start': '2017-06-02',
   'EncounterID': 'E001',
   'Encounter.reasonCode': 'Progressive leg tingling and burning',
   'patientId': 'P003',
   'Encounter.hospitalization.dischargeDisposition': 'Home',
   'Encounter.class': 'Ambulatory',
   'Observation[key]': 'Neurologic exam: patchy distal sensory loss and reduced vibration sense in both feet',
   'DiagnosticReport': 'NCS/EMG: multifocal demyelinating features in lower limb nerves',
   'NED_Entities': '[{\'source\': \'concat\', \'start\': 14, \'end\': 24, \'text\': \'leg tingling\', \'label\': \'Diseases of the nervous system\', \'assertion\': \'present\', \'temporality\': \'acute\', \'rationale\': \'Patient reports leg tingling.\', \'icd_id\': \'G57.1\', \'icd_label\': \'Meralgia paraesthetica\', \'confidence\': 0.95, \'linking_rationale\': "The mention of \'leg tingling\' aligns with the ICD-10 code G57.1 (Meralgia paraesthetica), which describes the tingling sensat

# Ontology in the Knowledge Graph

## HPO Example

In [23]:
graph_client.query("""
    MATCH (p:HpoPhenotype {label: "Abnormality of the endocrine system"})<-[:HAS_PHENOTYPIC_FEATURE]-(d:HpoDisease)
    RETURN  d.label AS disease
""")

[{'disease': 'Biemond syndrome II'},
 {'disease': 'Pseudovaginal perineoscrotal hypospadias'},
 {'disease': 'Ectodermal dysplasia with adrenal cyst'},
 {'disease': 'Thymic-Renal-Anal-Lung dysplasia'},
 {'disease': 'Myasthenia gravis'},
 {'disease': 'THIOUREA TASTINGPHENYLTHIOCARBAMIDE TASTING, INCLUDED'},
 {'disease': 'MENOPAUSE, NATURAL, AGE AT, QUANTITATIVE TRAIT LOCUS 1'},
 {'disease': 'Candidiasis, familial chronic mucocutaneous, autosomal dominant'},
 {'disease': 'PYGMY'},
 {'disease': 'PURA-related severe neonatal hypotonia-seizures-encephalopathy syndrome'},
 {'disease': 'Semilobar holoprosencephaly'},
 {'disease': 'Multiple endocrine neoplasia type 4'},
 {'disease': 'Fibrous dysplasia of bone'},
 {'disease': 'African trypanosomiasis'},
 {'disease': 'Midline interhemispheric variant of holoprosencephaly'},
 {'disease': 'Lobar holoprosencephaly'},
 {'disease': 'Alobar holoprosencephaly'},
 {'disease': 'Bardet-Biedl syndrome'},
 {'disease': 'Coccidioidomycosis'},
 {'disease': 'Iso

### RDFS (rdfs:subClassOf) Traversal

In [25]:
graph_client.query("""
MATCH (cat:HpoPhenotype {label: "Abnormality of the endocrine system"})
MATCH (sub)-[:subClassOf*0..]->(cat)
MATCH (dis)-[:HAS_PHENOTYPIC_FEATURE]->(sub)
MATCH (dis)-[:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype)
RETURN dis.label AS disease, collect(DISTINCT phe.label) AS features
ORDER BY size(features) ASC, disease
SKIP 100 LIMIT 5
""")

[{'disease': 'Cataract-deafness-hypogonadism syndrome',
  'features': ['Hypogonadism',
   'Sensorineural hearing impairment',
   'Developmental cataract',
   'Short stature',
   'Mild intellectual disability',
   'Generalized hypertrichosis']},
 {'disease': 'Congenital atransferrinemia',
  'features': ['Recurrent infections',
   'Hypothyroidism',
   'Anemia',
   'Arthritis',
   'Abnormality of the cardiovascular system',
   'Abnormality of the pancreas']},
 {'disease': 'Deafness, autosomal recessive 4, with enlarged vestibular aqueduct',
  'features': ['Autosomal recessive inheritance',
   'Sensorineural hearing impairment',
   'Incomplete partition of the cochlea type II',
   'Goiter',
   'Congenital onset',
   'Enlarged vestibular aqueduct']},
 {'disease': 'Diabetes mellitus, transient neonatal, 1',
  'features': ['Autosomal dominant inheritance',
   'Intrauterine growth retardation',
   'Severe failure to thrive',
   'Dehydration',
   'Hyperglycemia',
   'Transient neonatal diabetes

# Ontology Mapping

## Ontology Mapping Based on UMLS

In [27]:
graph_client.query("""
MATCH path = (d:IcdDisease)<-[:UMLS_TO_ICD]-(:Umls)-[:UMLS_TO_HPO_PHENOTYPE]->(p:HpoPhenotype)
WITH path, d, p,
     relationships(path) AS path_edges,
     nodes(path)         AS path_nodes
WITH
  // always get some string for each node
  [n IN path_nodes |
     COALESCE(n.label, n.id, elementId(n))
  ] AS node_names,
  [r IN path_edges | type(r)] AS rel_types,
  [r IN path_edges |
     COALESCE(
       startNode(r).label,
       startNode(r).id,
       elementId(startNode(r))
     )
  ] AS rel_starts,
  d, p
WITH [i IN range(0, size(node_names) - 1) |
  CASE
    WHEN i = size(node_names) - 1
      THEN '(' + node_names[i] + ')'
    WHEN node_names[i] = rel_starts[i]
      THEN '(' + node_names[i] + ')' + '-[:' + rel_types[i] + ']->'
    ELSE '(' + node_names[i] + ')' + '<-[:' + rel_types[i] + ']-'
  END
] AS string_paths, d, p
RETURN DISTINCT
  apoc.text.join(string_paths, '') AS `Extracted path`
LIMIT 10;
""")

[{'Extracted path': '(Paratyphoid fever, unspecified)<-[:UMLS_TO_ICD]-(C0015672)-[:UMLS_TO_HPO_PHENOTYPE]->(Fatigue)'},
 {'Extracted path': '(Other salmonella infections)<-[:UMLS_TO_ICD]-(C0085593)-[:UMLS_TO_HPO_PHENOTYPE]->(Chills)'},
 {'Extracted path': '(Shigellosis)<-[:UMLS_TO_ICD]-(C0015967)-[:UMLS_TO_HPO_PHENOTYPE]->(Fever)'},
 {'Extracted path': '(Enterocolitis due to Clostridium difficile)<-[:UMLS_TO_ICD]-(C0238106)-[:UMLS_TO_HPO_PHENOTYPE]->(Clostridium difficile colitis)'},
 {'Extracted path': '(Other bacterial foodborne intoxications, not elsewhere classified)<-[:UMLS_TO_ICD]-(C0231218)-[:UMLS_TO_HPO_PHENOTYPE]->(Malaise)'},
 {'Extracted path': '(Amoebiasis)<-[:UMLS_TO_ICD]-(C0039070)-[:UMLS_TO_HPO_PHENOTYPE]->(Syncope)'},
 {'Extracted path': '(Amoebiasis)<-[:UMLS_TO_ICD]-(C0039070)-[:UMLS_TO_HPO_PHENOTYPE]->(Loss of consciousness)'},
 {'Extracted path': '(Other protozoal intestinal diseases)<-[:UMLS_TO_ICD]-(C0009421)-[:UMLS_TO_HPO_PHENOTYPE]->(Coma)'},
 {'Extracted path': 

## Ontology Mapping with LLMs

### Candidate Selection

In [30]:
user_query = "Shortness of breath"
q_embed = api.post('/v1/embeddings', {'input': [user_query]})[0]['data'][0]['embedding']
k = 10

result = graph_client.query(
    """
    CALL db.index.vector.queryNodes('hpo_phenotype_embedding', $k, $qe)
        YIELD node, score
        RETURN node.id AS id, node.label AS label, score
        ORDER BY score DESC
        LIMIT $k
    """,
    {"k": k, "qe": q_embed},
)
for rec in result:
    print(f"{rec['score']:.3f}  {rec['label']}  (id={rec['id']})")

0.897  Dyspnea  (id=HP:0002094)
0.864  Respiratory distress  (id=HP:0002098)
0.851  Tachypnea  (id=HP:0002789)
0.822  Exertional dyspnea  (id=HP:0002875)
0.821  Breathing dysregulation  (id=HP:0005957)
0.818  Respiratory insufficiency  (id=HP:0002093)
0.814  Rest dyspnea  (id=HP:0033710)
0.808  Paroxysmal dyspnea  (id=HP:0012763)
0.802  Wheezing  (id=HP:0030828)
0.802  Abnormal pattern of respiration  (id=HP:0002793)


### Candidate Disambiguation

In [31]:
payload = {
    "source_concept": "R06.0 Dyspnoea",
    "source_context": """
    {
      "id": "R06.0",
      "name": "Dyspnoea",
      "parentName": "Abnormalities of breathing",
      "group":   { "groupName": "Symptoms and signs involving the circulatory and respiratory systems" },
      "chapter": { "chapterName": "Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified" }
    }
    """,
    "candidate_list": """
    [
      {
        "score": 0.897,
        "id": "HP:0002094",
        "label": "Dyspnea",
        "exactSynonym": [
          "Shortness of breath",
          "Breathlessness"
        ],
        "description": "A subjective sensation of difficult or uncomfortable breathing."
      },
      {
        "score": 0.865,
        "id": "HP:0002098",
        "label": "Respiratory distress",
        "exactSynonym": [
          "Distressed breathing"
        ],
        "description": "A clinical state characterized by labored or difficult breathing that may be accompanied by use of accessory muscles."
      },
      {
        "score": 0.852,
        "id": "HP:0002789",
        "label": "Tachypnea",
        "exactSynonym": [
          "Increased respiratory rate",
          "Rapid breathing"
        ],
        "description": "An abnormally high respiratory rate for age at rest."
      },
      {
        "score": 0.822,
        "id": "HP:0005957",
        "label": "Breathing dysregulation",
        "exactSynonym": [
          "Disordered breathing regulation"
        ],
        "description": "Abnormal control or patterning of the respiratory rhythm."
      },
    ]
    """,
}


tool_output = build_ontology_mapper_tool(chat_client).invoke(payload)
show_json(tool_output)

/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=OntologyMappingResponse(b...antically equivalent.')), input_type=OntologyMappingResponse])
  return self.__pydantic_serializer__.to_python(


```json
{
  "best_id": "HP:0002094",
  "best_label": "Dyspnea",
  "confidence": 0.9,
  "rationale": "The source concept 'Dyspnoea' is a direct synonym of 'Dyspnea' in the HPPO ontology.",
  "support": {
    "evidence": "Direct synonym in HPPO ontology.",
    "reason": "The source concept and the candidate concept are semantically equivalent."
  }
}
```

In [32]:
payload = {
    "source_concept": "R07.1 Chest pain on breathing",
    "source_context": """
    {
      "id": "R07.1",
      "name": "Chest pain on breathing",
      "parentName": "Pain in throat and chest",
      "group":   { "groupName": "Symptoms and signs involving the circulatory and respiratory systems" },
      "chapter": { "chapterName": "Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified" }
    }
    """,
    "candidate_list": """
    [
      {
        "score": 0.93,
        "id": "HP:0002104",
        "label": "Chest pain",
        "exactSynonym": [
          "Thoracic pain"
        ],
        "description": "Pain localized to the anterior or posterior chest wall, without specification of the precipitating factor, temporal pattern, or relationship to respiration or exertion. This term is intentionally broad and may encompass musculoskeletal, cardiac, pulmonary, gastrointestinal, or idiopathic etiologies when the clinical context is not further specified."
      },
      {
        "score": 0.90,
        "id": "HP:0100749",
        "label": "Exertional chest pain",
        "exactSynonym": [
          "Chest pain on exertion"
        ],
        "description": "Chest discomfort or pain that is primarily triggered, precipitated, or worsened by physical activity or emotional stress and tends to improve with rest. This feature is classically associated with myocardial ischemia or other cardiopulmonary limitations related to increased workload, rather than with respiratory movements such as inspiration or coughing."
      },
      {
        "score": 0.87,
        "id": "HP:0030165",
        "label": "Pleuritic chest pain",
        "exactSynonym": [
          "Pleural pain"
        ],
        "description": "A sharp, stabbing, or burning chest pain that is characteristically exacerbated by deep inspiration, coughing, sneezing, or other movements of the chest wall and diaphragm, and often relieved by lying on the affected side. This symptom is typically associated with inflammation or irritation of the pleura (e.g., pleuritis, pulmonary embolism, or pneumonia) and corresponds clinically to chest pain that occurs specifically on breathing."
      }
    ]
    """,
}


tool_output = build_ontology_mapper_tool(chat_client).invoke(payload)
show_json(tool_output)

/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=OntologyMappingResponse(b...n the candidate list.')), input_type=OntologyMappingResponse])
  return self.__pydantic_serializer__.to_python(


```json
{
  "best_id": "HP:0030165",
  "best_label": "Pleuritic chest pain",
  "confidence": 0.93,
  "rationale": "The source concept describes chest pain on breathing, which is specifically exacerbated by deep inspiration, coughing, sneezing, or other movements of the chest wall and diaphragm. This is the most specific and semantically equivalent concept in the candidate list.",
  "support": {
    "evidence": "The description of 'Pleuritic chest pain' directly aligns with the source concept's description of chest pain on breathing.",
    "reason": "The source concept describes chest pain on breathing, which is specifically exacerbated by deep inspiration, coughing, sneezing, or other movements of the chest wall and diaphragm. This is the most specific and semantically equivalent concept in the candidate list."
  }
}
```

### Ontology Mapping Results

In [ ]:
result =graph_client.query("""
    MATCH (d:IcdDisease)-[r:ICD_MAPS_TO_HPO_BY_EMBEDDING]->(p:HpoPhenotype)
    RETURN d.id as icd_id,
           d.label as icd_label,
           p.id AS hpo_id,
           p.label AS hpo_label,
           r.confidence AS confidence,
           r.evidence AS evidence,
           r.rationale AS rationale
    OFFSET 100
    LIMIT 1
""")
for rec in result:
    print(f"(ICD ID: {rec['icd_id']} ({rec['icd_label']})\n(HPO ID: {rec['hpo_id']}) ({rec['hpo_label']})\n(evidence={rec['evidence']}\nconfidence={rec['confidence']}\nrationale={rec['rationale']})")

In [ ]:
graph_client.query("""
    MATCH path = (d:IcdDisease)<-[:UMLS_TO_ICD]-(:UMLS)-[:UMLS_TO_HPO_PHENOTYPE]->(p:HpoPhenotype)
    WHERE d.id = 'A18.3' and p.id = 'HP:0034841'
    RETURN path
""")

# GNN-Enhanced Ontology Mapping

## The Disambiguation Problem

Vector search retrieves semantically similar HPO candidates, but struggles to distinguish between them when
multiple terms share lexical overlap with the query. For example:

**ICD M54.2 — Cervicalgia** (neck pain)

Qwen retrieves top-20 HPO candidates. Many are generic pain terms — "Pain", "Arthralgia", "Myalgia" — that
rank highest because they share the most semantic overlap with "Cervicalgia." The correct answer,
**"Cervical radiculopathy"** (HP:0002816), is buried at rank 19 because its HPO description emphasizes nerve
root compression rather than the pain symptom.

## GNN Reranking

The **ResidualGNN** aggregates ontology hierarchy context via GAT message passing. It enriches each node's
embedding with information about *where* it sits in its ontology tree:

- **"Cervical radiculopathy"** sits under `Peripheral neuropathy → Abnormality of the nervous system` — structurally close to cervical/spinal phenotypes
- **"Pain"** sits under `Constitutional symptom` — a generic branch far from musculoskeletal specifics
- **M54.2** sits under `Dorsopathies → Other dorsopathies` in the ICD hierarchy

This structural context breaks the semantic tie. GNN-refined similarity pushes "Cervical radiculopathy"
from rank 19 → rank 3, making it available for LLM disambiguation.

## Aggregate Results (from training evaluation)

- **368 test ICD codes**, 34 Qwen failures (correct HPO ranked > 5)
- **17/34 (50%) rescued** to top-5 by GNN reranking
- **+0.028 Recall@5** improvement overall
- Notable rescues: Q76.6 (rank 11→1), C18.9 (rank 7→1), M54.2 (rank 19→3)

### GNN API Connector

In [33]:
url_gnn = load_config_api("gnn", path="../config.ini")
gnn_api = ApiClient(url_gnn)

# Health check
body, status, _ = gnn_api.get('/healthz')
print(f"GNN server: status={status}")
print(f"  Model: {body['model']}, alpha={body['alpha']}")
print(f"  Nodes: {body['nodes']}, Edges: {body['edges']}")

GNN server: status=200
  Model: ResidualGNN, alpha=0.102
  Nodes: 45205, Edges: 330488


### Hard Case: Qwen Fails, GNN Rescues

In [35]:
# Hard case: M54.2 Cervicalgia — Qwen buries the correct HPO in generic pain terms
icd_code = "M54.2"

# Step 1: Qwen vector search for top-20 HPO candidates
icd_label = graph_client.query(
    "MATCH (d:IcdDisease {id: $code}) RETURN d.label AS label",
    {"code": icd_code}
)[0]["label"]

q_embed = api.post('/v1/embeddings', {'input': [icd_label]})[0]['data'][0]['embedding']
qwen_candidates = graph_client.query("""
    CALL db.index.vector.queryNodes('hpo_phenotype_embedding', $k, $qe)
    YIELD node, score
    RETURN node.id AS id, node.label AS label, score
    ORDER BY score DESC
    LIMIT $k
""", {"k": 20, "qe": q_embed})

print(f"ICD {icd_code}: {icd_label}")
print(f"Qwen top-20 candidates retrieved.\n")

# Step 2: GNN rerank same candidates
candidate_codes = [c['id'] for c in qwen_candidates]
gnn_result, _, _ = gnn_api.post('/compare', {
    'icd_code': icd_code,
    'candidate_codes': candidate_codes,
    'top_k': 10,
})

# Step 3: Side-by-side comparison
print(f"{'Rank':<6} {'Qwen':>6} {'GNN':>6} {'Delta':>6}  {'HPO Code':<12} {'Label'}")
print("-" * 80)
for c in gnn_result['by_gnn']:
    delta_str = f"+{c['rank_delta']}" if c['rank_delta'] > 0 else str(c['rank_delta'])
    print(f"  GNN {c['gnn_rank']:>2}  Q {c['qwen_rank']:>2}  {delta_str:>5}   {c['code']:<12} {c['label']}")

ICD M54.2: Cervicalgia
Qwen top-20 candidates retrieved.

Rank     Qwen    GNN  Delta  HPO Code     Label
--------------------------------------------------------------------------------
  GNN  1  Q 19    +18   HP:0030833   Neck pain
  GNN  2  Q  1     -1   HP:0008480   Cervical spondylosis
  GNN  3  Q  3      0   HP:0030009   Cervical insufficiency
  GNN  4  Q  9     +5   HP:0002947   Cervical kyphosis
  GNN  5  Q  6     +1   HP:0008445   Cervical spinal canal stenosis
  GNN  6  Q  2     -4   HP:6000107   Cervical motion tenderness
  GNN  7  Q 20    +13   HP:0000473   Torticollis
  GNN  8  Q  8      0   HP:0008462   Cervical instability
  GNN  9  Q  5     -4   HP:0003308   Cervical subluxation
  GNN 10  Q 12     +2   HP:0012318   Occipital neuralgia


### Easy Case: Qwen Succeeds, GNN Preserves

In [37]:
# Easy case: R06.0 Dyspnoea — Qwen already ranks the correct HPO at #1
icd_code = "R06.0"

icd_label = graph_client.query(
    "MATCH (d:IcdDisease {id: $code}) RETURN d.label AS label",
    {"code": icd_code}
)[0]["label"]

q_embed = api.post('/v1/embeddings', {'input': [icd_label]})[0]['data'][0]['embedding']
qwen_candidates = graph_client.query("""
    CALL db.index.vector.queryNodes('hpo_phenotype_embedding', $k, $qe)
    YIELD node, score
    RETURN node.id AS id, node.label AS label, score
    ORDER BY score DESC
    LIMIT $k
""", {"k": 20, "qe": q_embed})

print(f"ICD {icd_code}: {icd_label}")
print(f"Qwen top-20 candidates retrieved.\n")

candidate_codes = [c['id'] for c in qwen_candidates]
gnn_result, _, _ = gnn_api.post('/compare', {
    'icd_code': icd_code,
    'candidate_codes': candidate_codes,
    'top_k': 10,
})

print(f"{'Rank':<6} {'Qwen':>6} {'GNN':>6} {'Delta':>6}  {'HPO Code':<12} {'Label'}")
print("-" * 80)
for c in gnn_result['by_gnn']:
    delta_str = f"+{c['rank_delta']}" if c['rank_delta'] > 0 else str(c['rank_delta'])
    print(f"  GNN {c['gnn_rank']:>2}  Q {c['qwen_rank']:>2}  {delta_str:>5}   {c['code']:<12} {c['label']}")

ICD R06.0: Dyspnoea
Qwen top-20 candidates retrieved.

Rank     Qwen    GNN  Delta  HPO Code     Label
--------------------------------------------------------------------------------
  GNN  1  Q  1      0   HP:0002094   Dyspnea
  GNN  2  Q  3     +1   HP:0002789   Tachypnea
  GNN  3  Q  5     +2   HP:0002098   Respiratory distress
  GNN  4  Q 15    +11   HP:0002883   Hyperventilation
  GNN  5  Q  2     -3   HP:0002875   Exertional dyspnea
  GNN  6  Q  7     +1   HP:0012764   Orthopnea
  GNN  7  Q  9     +2   HP:0002093   Respiratory insufficiency
  GNN  8  Q 17     +9   HP:0002791   Hypoventilation
  GNN  9  Q  8     -1   HP:0005957   Breathing dysregulation
  GNN 10  Q 16     +6   HP:0005943   Respiratory arrest


### GNN-Enhanced LLM Disambiguation

Use GNN-reranked candidates as input to the LLM ontology mapper.
In the hard case, the correct HPO is now in the top-4 candidate set.

In [40]:
# Qwen-only disambiguation for M54.2 (no GNN reranking)
icd_code = "M54.2"

icd_label = graph_client.query(
    "MATCH (d:IcdDisease {id: $code}) RETURN d.label AS label",
    {"code": icd_code}
)[0]["label"]

icd_context = graph_client.query("""
    MATCH (d:IcdDisease {id: $code})
    OPTIONAL MATCH (d)<-[:HAS_CHILD]-(parent:IcdDisease)
    OPTIONAL MATCH (d)<-[:GROUP_HAS_DISEASE]-(g:IcdGroup)
    OPTIONAL MATCH (g)<-[:CHAPTER_HAS_DISEASE]-(ch:IcdChapter)
    RETURN d.label AS name,
           parent.label AS parentName,
           g.label AS groupName,
           ch.label AS chapterName
    LIMIT 1
""", {"code": icd_code})[0]

# Vector search top-4 only (no GNN)
q_embed = api.post('/v1/embeddings', {'input': [icd_label]})[0]['data'][0]['embedding']
qwen_top4 = graph_client.query("""
    CALL db.index.vector.queryNodes('hpo_phenotype_embedding', $k, $qe)
    YIELD node, score
    RETURN node.id AS id, node.label AS label, score
    ORDER BY score DESC
    LIMIT $k
""", {"k": 4, "qe": q_embed})

# Fetch HPO details
top4_codes = [c['id'] for c in qwen_top4]
hpo_details = graph_client.query("""
    UNWIND $codes AS code
    MATCH (h:HpoPhenotype {id: code})
    RETURN h.id AS id, h.label AS label,
           h.IAO_0000115 AS description,
           h.hasExactSynonym AS exactSynonym
""", {"codes": top4_codes})

import json

candidate_list = []
for hpo in hpo_details:
    qwen_entry = next(c for c in qwen_top4 if c['id'] == hpo['id'])
    candidate_list.append({
        "score": qwen_entry["score"],
        "id": hpo["id"],
        "label": hpo["label"],
        "exactSynonym": hpo.get("exactSynonym", []) or [],
        "description": hpo.get("description", "") or "",
    })

payload = {
    "source_concept": f"{icd_code} {icd_label}",
    "source_context": json.dumps({
        "id": icd_code,
        "name": icd_context["name"],
        "parentName": icd_context.get("parentName", ""),
        "group": {"groupName": icd_context.get("groupName", "")},
        "chapter": {"chapterName": icd_context.get("chapterName", "")},
    }),
    "candidate_list": json.dumps(candidate_list),
}

print(f"Source: {icd_code} {icd_label}")
print(f"Qwen top-4 candidates (no GNN):")
for c in candidate_list:
    print(f"  {c['score']:.4f}  {c['id']}  {c['label']}")

print("\nLLM disambiguation:")
tool_output = build_ontology_mapper_tool(chat_client).invoke(payload)
show_json(tool_output)


Source: M54.2 Cervicalgia
Qwen top-4 candidates (no GNN):
  0.8791  HP:0008480  Cervical spondylosis
  0.8791  HP:6000107  Cervical motion tenderness
  0.8655  HP:0030009  Cervical insufficiency
  0.8551  HP:0032535  Cervical (neck)

LLM disambiguation:


/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=OntologyMappingResponse(b...cause of cervicalgia.')), input_type=OntologyMappingResponse])
  return self.__pydantic_serializer__.to_python(


```json
{
  "best_id": "HP:0008480",
  "best_label": "Cervical spondylosis",
  "confidence": 0.879112,
  "rationale": "Cervicalgia is a general term for neck pain. Cervical spondylosis is a degenerative condition affecting the cervical spine, which is a more specific and semantically equivalent concept.",
  "support": {
    "evidence": "Semantic equivalence and clinical relevance.",
    "reason": "Cervical spondylosis is a common cause of cervicalgia."
  }
}
```

In [39]:
# Rerank M54.2 candidates via GNN and feed top-4 to LLM
icd_code = "M54.2"

icd_label = graph_client.query(
    "MATCH (d:IcdDisease {id: $code}) RETURN d.label AS label",
    {"code": icd_code}
)[0]["label"]

# Get ICD context from graph
icd_context = graph_client.query("""
    MATCH (d:IcdDisease {id: $code})
    OPTIONAL MATCH (d)<-[:HAS_CHILD]-(parent:IcdDisease)
    OPTIONAL MATCH (d)<-[:GROUP_HAS_DISEASE]-(g:IcdGroup)
    OPTIONAL MATCH (g)<-[:CHAPTER_HAS_DISEASE]-(ch:IcdChapter)
    RETURN d.label AS name,
           parent.label AS parentName,
           g.label AS groupName,
           ch.label AS chapterName
    LIMIT 1
""", {"code": icd_code})[0]

# Vector search for top-20 candidates (self-contained)
q_embed = api.post('/v1/embeddings', {'input': [icd_label]})[0]['data'][0]['embedding']
qwen_candidates = graph_client.query("""
    CALL db.index.vector.queryNodes('hpo_phenotype_embedding', $k, $qe)
    YIELD node, score
    RETURN node.id AS id, node.label AS label, score
    ORDER BY score DESC
    LIMIT $k
""", {"k": 20, "qe": q_embed})

# Get GNN-reranked top-4 candidates with their HPO details
candidate_codes = [c['id'] for c in qwen_candidates]
gnn_reranked, _, _ = gnn_api.post('/rerank', {
    'icd_code': icd_code,
    'candidate_codes': candidate_codes,
    'top_k': 4,
})

# Fetch HPO details for the top-4
top4_codes = [c['code'] for c in gnn_reranked['candidates']]
hpo_details = graph_client.query("""
    UNWIND $codes AS code
    MATCH (h:HpoPhenotype {id: code})
    RETURN h.id AS id, h.label AS label,
           h.IAO_0000115 AS description,
           h.hasExactSynonym AS exactSynonym
""", {"codes": top4_codes})

# Build candidate list JSON for the mapper
import json

candidate_list = []
for hpo in hpo_details:
    gnn_entry = next(c for c in gnn_reranked['candidates'] if c['code'] == hpo['id'])
    candidate_list.append({
        "score": gnn_entry["gnn_score"],
        "id": hpo["id"],
        "label": hpo["label"],
        "exactSynonym": hpo.get("exactSynonym", []) or [],
        "description": hpo.get("description", "") or "",
    })

payload = {
    "source_concept": f"{icd_code} {icd_label}",
    "source_context": json.dumps({
        "id": icd_code,
        "name": icd_context["name"],
        "parentName": icd_context.get("parentName", ""),
        "group": {"groupName": icd_context.get("groupName", "")},
        "chapter": {"chapterName": icd_context.get("chapterName", "")},
    }),
    "candidate_list": json.dumps(candidate_list),
}

print(f"Source: {icd_code} {icd_label}")
print(f"GNN-reranked top-4 candidates:")
for c in candidate_list:
    print(f"  {c['score']:.4f}  {c['id']}  {c['label']}")

print("\nLLM disambiguation:")
tool_output = build_ontology_mapper_tool(chat_client).invoke(payload)
show_json(tool_output)


Source: M54.2 Cervicalgia
GNN-reranked top-4 candidates:
  0.6241  HP:0030833  Neck pain
  0.6202  HP:0008480  Cervical spondylosis
  0.5951  HP:0030009  Cervical insufficiency
  0.5887  HP:0002947  Cervical kyphosis

LLM disambiguation:


/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=OntologyMappingResponse(b...aning of cervicalgia.")), input_type=OntologyMappingResponse])
  return self.__pydantic_serializer__.to_python(


```json
{
  "best_id": "HP:0030833",
  "best_label": "Neck pain",
  "confidence": 0.7,
  "rationale": "Neck pain is a common symptom of cervicalgia and is semantically equivalent.",
  "support": {
    "evidence": "Neck pain is a common symptom of cervicalgia.",
    "reason": "The description of 'Neck pain' aligns with the clinical meaning of cervicalgia."
  }
}
```

### Why GNN Reranking Works: Structural Context

The 2-layer GAT aggregates information from each node's 2-hop neighborhood. For the M54.2 (Cervicalgia) hard case:

- **Neck pain** (HP:0030833) — 21 HpoDisease links, reaching 523 co-occurring phenotypes at hop 2. This rich neighborhood overlaps with the neurological/musculoskeletal context of dorsalgia.
- **Cervical spondylosis** (HP:0008480) — only 4 HpoDisease links and 82 co-occurring phenotypes. Despite sharing "cervical" lexically, it sits in a narrow skeletal morphology branch.

Qwen embeddings capture lexical similarity ("cervical" → "cervical spondylosis"), but the GAT-refined embeddings incorporate structural context — the diseases each phenotype co-occurs with, and their broader ontology neighborhoods. This breaks the semantic tie in favor of the clinically correct mapping.


In [42]:
# What the GNN sees: hierarchy + disease links + 2-hop neighborhood
icd_code = "M54.2"
correct_hpo = "HP:0030833"   # Neck pain (Qwen rank 19 → GNN rank 1)
top_qwen_hpo = "HP:0008480"  # Cervical spondylosis (Qwen rank 1)

# --- ICD hierarchy ---
icd_hierarchy = graph_client.query("""
    MATCH (d:IcdDisease {id: $code})
    OPTIONAL MATCH (d)<-[:HAS_CHILD]-(parent:IcdDisease)
    OPTIONAL MATCH (d)<-[:GROUP_HAS_DISEASE]-(g:IcdGroup)
    OPTIONAL MATCH (g)<-[:CHAPTER_HAS_DISEASE]-(ch:IcdChapter)
    RETURN d.label AS label, parent.label AS parent,
           g.label AS group, ch.label AS chapter
""", {"code": icd_code})[0]

print(f"=== ICD Hierarchy: {icd_code} ===")
print(f"  Chapter: {icd_hierarchy['chapter']}")
print(f"  Group:   {icd_hierarchy['group']}")
print(f"  Parent:  {icd_hierarchy['parent']}")
print(f"  Disease: {icd_hierarchy['label']}")

# --- HPO subClassOf chains ---
for code, label, rank_info in [
    (correct_hpo, "Neck pain", "GNN rank 1, Qwen rank 19"),
    (top_qwen_hpo, "Cervical spondylosis", "Qwen rank 1"),
]:
    paths = graph_client.query("""
        MATCH path = (p:HpoPhenotype {id: $code})-[:subClassOf*]->(anc:HpoPhenotype)
        WHERE NOT (anc)-[:subClassOf]->()
        RETURN [n IN nodes(path) | n.label] AS hierarchy
    """, {"code": code})

    print(f"\n=== HPO Hierarchy: {code} ({label}) — {rank_info} ===")
    for i, p in enumerate(paths):
        print(f"  Path {i+1}: {' → '.join(p['hierarchy'])}")

# --- HpoDisease nodes linked via HAS_PHENOTYPIC_FEATURE ---
for code, label in [(correct_hpo, "Neck pain"), (top_qwen_hpo, "Cervical spondylosis")]:
    diseases = graph_client.query("""
        MATCH (d:HpoDisease)-[:HAS_PHENOTYPIC_FEATURE]->(p:HpoPhenotype {id: $code})
        RETURN d.label AS disease
        LIMIT 10
    """, {"code": code})

    print(f"\n=== HpoDisease nodes linked to {label} (GAT layer 1 message sources) ===")
    for d in diseases:
        print(f"  {d['disease']}")

# --- 2-hop neighborhood sizes (what 2-layer GAT aggregates) ---
print("\n=== 2-Layer GAT Neighborhood Comparison ===")
for code, label in [(correct_hpo, "Neck pain"), (top_qwen_hpo, "Cervical spondylosis")]:
    stats = graph_client.query("""
        MATCH (p:HpoPhenotype {id: $code})
        OPTIONAL MATCH (p)-[:subClassOf]->(parent)
        OPTIONAL MATCH (sibling)-[:subClassOf]->(parent)
        OPTIONAL MATCH (d:HpoDisease)-[:HAS_PHENOTYPIC_FEATURE]->(p)
        OPTIONAL MATCH (d)-[:HAS_PHENOTYPIC_FEATURE]->(co_phe:HpoPhenotype)
        RETURN count(DISTINCT parent) AS parents,
               count(DISTINCT sibling) AS siblings,
               count(DISTINCT d) AS linked_diseases,
               count(DISTINCT co_phe) AS co_phenotypes_hop2
    """, {"code": code})[0]

    print(f"\n  {code} ({label}):")
    print(f"    Hop 1: {stats['parents']} parent(s), {stats['siblings']} siblings, {stats['linked_diseases']} HpoDisease links")
    print(f"    Hop 2: {stats['co_phenotypes_hop2']} co-occurring phenotypes (via shared diseases)")


=== ICD Hierarchy: M54.2 ===
  Chapter: None
  Group:   None
  Parent:  Dorsalgia
  Disease: Cervicalgia

=== HPO Hierarchy: HP:0030833 (Neck pain) — GNN rank 1, Qwen rank 19 ===
  Path 1: Neck pain → Pain in head and neck region → Pain → Constitutional symptom → Phenotypic abnormality → All

=== HPO Hierarchy: HP:0008480 (Cervical spondylosis) — Qwen rank 1 ===
  Path 1: Cervical spondylosis → Abnormal cervical spine morphology → Abnormality of the cervical spine → Abnormality of the vertebral column → Abnormal axial skeleton morphology → Abnormal skeletal morphology → Abnormality of the skeletal system → Abnormality of the musculoskeletal system → Phenotypic abnormality → All
  Path 2: Cervical spondylosis → Abnormal cervical spine morphology → Abnormal vertebral morphology → Abnormality of the vertebral column → Abnormal axial skeleton morphology → Abnormal skeletal morphology → Abnormality of the skeletal system → Abnormality of the musculoskeletal system → Phenotypic abnormality →

# Patient Annotation

### Named Entity Recognition

DO NOT TRUST THE CHARACTER INDEX!

In [43]:
payload = {
    "patient_id": "patient_002",
    "encounter_id": "enc_145",
    "icd_chapters": [
        "Certain infectious and parasitic diseases",
        "Neoplasms",
        "Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism",
        "Endocrine, nutritional and metabolic diseases",
        "Mental and behavioural disorders",
        "Diseases of the nervous system",
        "Diseases of the eye and adnexa",
        "Diseases of the ear and mastoid process",
        "Diseases of the circulatory system",
        "Diseases of the respiratory system",
        "Diseases of the digestive system",
        "Diseases of the skin and subcutaneous tissue",
        "Diseases of the musculoskeletal system and connective tissue",
        "Diseases of the genitourinary system",
        "Pregnancy, childbirth and the puerperium",
        "Certain conditions originating in the perinatal period",
        "Congenital malformations, deformations and chromosomal abnormalities",
        "Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified",
        "Injury, poisoning and certain other consequences of external causes",
        "External causes of morbidity and mortality",
        "Factors influencing health status and contact with health services",
        "Codes for special purposes"
    ],
    "concat_text": (
        "Abdominal pain | "
        "Shortness of breath and chest tightness | "
        "Episodic dizziness and palpitations | "
        "Hypertension, type 2 diabetes mellitus, and iron-deficiency anemia | "
        "No Chest pain"
    ),
    "narrative_text": (
        "No narrative text provided."
    )
}


tool_output = build_patient_ner_tool(chat_client).invoke(payload)
show_json(tool_output)

/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=PatientNERResponse(patien...ed in the narrative.')]), input_type=PatientNERResponse])
  return self.__pydantic_serializer__.to_python(


```json
{
  "patient_id": "patient_002",
  "encounter_id": "enc_145",
  "entities": [
    {
      "source": "concat",
      "start": 14,
      "end": 26,
      "text": "Abdominal pain",
      "label": "Diseases of the digestive system",
      "assertion": "present",
      "temporality": "unspecified",
      "rationale": "Present in the concatenated text."
    },
    {
      "source": "concat",
      "start": 30,
      "end": 42,
      "text": "Shortness of breath and chest tightness",
      "label": "Diseases of the respiratory system",
      "assertion": "present",
      "temporality": "unspecified",
      "rationale": "Present in the concatenated text."
    },
    {
      "source": "concat",
      "start": 46,
      "end": 68,
      "text": "Episodic dizziness and palpitations",
      "label": "Diseases of the nervous system",
      "assertion": "present",
      "temporality": "unspecified",
      "rationale": "Present in the concatenated text."
    },
    {
      "source": "concat",
      "start": 72,
      "end": 94,
      "text": "Hypertension, type 2 diabetes mellitus, and iron-deficiency anemia",
      "label": "Endocrine, nutritional and metabolic diseases",
      "assertion": "present",
      "temporality": "unspecified",
      "rationale": "Present in the concatenated text."
    },
    {
      "source": "concat",
      "start": 103,
      "end": 105,
      "text": "No Chest pain",
      "label": "Diseases of the circulatory system",
      "assertion": "negated",
      "temporality": "unspecified",
      "rationale": "The patient denies chest pain."
    },
    {
      "source": "narrative",
      "start": 0,
      "end": 13,
      "text": "No narrative text provided.",
      "label": "Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified",
      "assertion": "unspecified",
      "temporality": "unspecified",
      "rationale": "No narrative text provided. This is a placeholder for a potential symptom or sign that may be present but not explicitly mentioned in the narrative."
    }
  ]
}
```

### Named Entity Disambiguation

In [44]:
ned_payload = {
    "mention": {
        "source": "concat",
        "start": 245,
        "end": 260,
        "text": "terminal ileitis",
        "label": "Diseases of the digestive system",
        "assertion": "present",
        "temporality": "chronic",
        "rationale": (
            "Imaging and colonoscopy describe inflammation of the terminal ileum. "
            "Clinical context (chronic diarrhea, weight loss) "
            "suggests inflammatory bowel disease rather than acute infection."
        ),
    },
    "candidates": [
        {
            "score": 0.912,
            "id": "K52.9",
            "label": "Noninfective gastroenteritis and colitis, unspecified"
        },
        {
            "score": 0.887,
            "id": "A09.0",
            "label": "Infectious gastroenteritis and colitis, unspecified"
        },
        {
            "score": 0.871,
            "id": "K52.0",
            "label": "Gastroenteritis and colitis due to radiation"
        },
        {
            "score": 0.842,
            "id": "K50.00",
            "label": "Crohn's disease of small intestine without complications"
        },
        {
            "score": 0.824,
            "id": "K50.80",
            "label": "Crohn's disease of both small and large intestine without complications"
        },
        {
            "score": 0.801,
            "id": "K51.90",
            "label": "Ulcerative colitis, unspecified, without complications"
        },
    ],
    "other_mentions": [
        {
            "text": "long-standing Crohn disease diagnosed at age 19",
            "label": "Diseases of the digestive system",
        },
        {
            "text": "skip lesions in terminal ileum and ascending colon on colonoscopy",
            "label": "Diseases of the digestive system",
        },
        {
            "text": "chronic watery diarrhea and 7 kg unintentional weight loss in 6 months",
            "label": "Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified",
        },
        {
            "text": "non-caseating granulomas on ileal biopsy",
            "label": "Diseases of the digestive system",
        },
    ],
}

# Invoke the Patient NED tool
ned_tool_output = build_patient_ned_tool(chat_client).invoke(ned_payload)
show_json(ned_tool_output)


/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=PatientNEDResponse(source...by the other mentions.'), input_type=PatientNEDResponse])
  return self.__pydantic_serializer__.to_python(


```json
{
  "source": "concat",
  "start": 245,
  "end": 260,
  "text": "terminal ileitis",
  "label": "Diseases of the digestive system",
  "assertion": "present",
  "temporality": "chronic",
  "rationale": "The mention describes inflammation of the terminal ileum, which is consistent with terminal ileitis. The chronic nature of the condition is also supported by the other mentions.",
  "icd_id": "K50.00",
  "icd_label": "Crohn's disease of small intestine without complications",
  "confidence": 0.95,
  "linking_rationale": "The mention describes inflammation of the terminal ileum, which is consistent with terminal ileitis. The chronic nature of the condition is also supported by the other mentions."
}
```

## Question and Answers

### General Medical Questions (Ontology-Driven)

In [45]:
questions = [
    #"For cystic fibrosis, what symptoms are documented and what is the evidence supporting them?",
    #"Which phenotypes of Marfan syndrome have published clinical study evidence, and what are their frequencies?",
    #"For Duchenne muscular dystrophy, list all associated symptoms and include the supporting publications.",
    "Which diseases are associated with short stature? Show me the reported frequencies.",
    #"List all phenotypes of Rett syndrome with their frequencies and PubMed URLs.",
    # "Which diseases are linked to the phenotype 'muscle weakness' supported by published clinical studies?",
    #"Find diseases associated with 'recurrent infections' and show the evidence descriptions.",
    #"Which diseases have phenotypes with adult onset and what sources support them?", # Useful to show different use cases
    #"Which diseases are treated with insulin?",
    "Find phenotypes associated with the avian influenza disease",
    # "For the ICD disease 'Typhoid fever', what HPO phenotypes are mapped and with what confidence?",
    "What is the average confidence of all HPO mappings to ICD for Cholera?",
    "List diseases associated with recurrent fever supported by published clinical studies only.",
]

for q in questions:
    print("\n=== Question ===")
    print(q)
    general_medical_tool_output = build_general_medical_tool(chat_client, debug=True).invoke({
        "question": q,
        "top_k": 10
    })
    print("=== Output ===")
    print(general_medical_tool_output["explanation"])



=== Question ===
Which diseases are associated with short stature? Show me the reported frequencies.
=== text2cypher_pipeline: START ===
Question: 'Which diseases are associated with short stature? Show me the reported frequencies.'
Step 1: Generated Cypher:
MATCH (p:HpoPhenotype)
WHERE toLower(p.label) CONTAINS toLower("short stature")
       OR toLower(p.hasExactSynonym) CONTAINS toLower("short stature")
       OR toLower(p.hasNarrowSynonym) CONTAINS toLower("short stature")
MATCH (d:HpoDisease)-[r:HAS_PHENOTYPIC_FEATURE]->(p)
RETURN
     d.id AS disease_id,
     d.label AS disease_label,
     p.id AS hpo_id,
     p.label AS hpo_label,
     p.comment AS hpo_comment,
     r.frequency AS frequency,
     r.source AS source,
     r.url AS pubmed_url
ORDER BY d.label
Step 2: EXPLAIN on generated query: OK
Step 3: Relationship correction: no change


/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ValidateCypherOutput(errors=[]), input_type=ValidateCypherOutput])
  return self.__pydantic_serializer__.to_python(


Step 4: LLM validation: no errors found
Step 5: No correction needed
Step 6: Executing final Cypher query:
MATCH (p:HpoPhenotype)
WHERE toLower(p.label) CONTAINS toLower("short stature")
       OR toLower(p.hasExactSynonym) CONTAINS toLower("short stature")
       OR toLower(p.hasNarrowSynonym) CONTAINS toLower("short stature")
MATCH (d:HpoDisease)-[r:HAS_PHENOTYPIC_FEATURE]->(p)
RETURN
     d.id AS disease_id,
     d.label AS disease_label,
     p.id AS hpo_id,
     p.label AS hpo_label,
     p.comment AS hpo_comment,
     r.frequency AS frequency,
     r.source AS source,
     r.url AS pubmed_url
ORDER BY d.label
Step 6: Query returned 2268 row(s)
=== text2cypher_pipeline: END ===
=== Output ===
Answer:
According to the ontology, the following diseases are associated with short stature:

*   **12q14 microdeletion syndrome (ORPHA:94063, DECIPHER:76):** This phenotype is defined as "Short stature" (HP:0004322). The frequency is HP:0040281.
*   **13q12.3 microdeletion syndrome (ORPHA:41

### Patient Tool (Virtualized Resources)

In [46]:
patient_payload = {
    "patient_id": "P003",
    #"question": "Provide a concise clinical summary of patient P003 using all documented findings.",
    "question": "Provide me details on the follow up plan of patient P003 in the latest encounter.",
}

patient_tool_output = build_patient_info_tool(chat_client).invoke(patient_payload)
patient_tool_output

{'patient_id': 'P003',
 'question': 'Provide me details on the follow up plan of patient P003 in the latest encounter.',
 'encounter_date': 'latest',
 'has_data': True,
 'answer': 'Answer:\nThe follow-up plan for patient P003 in the latest encounter on 2026-03-03 is to monitor for new skin changes and reassess if the lesions spread. The patient is instructed to use regular emollients.\n',
 'raw_patient_view': [{'patient_id': 'P003',
   'identity': None,
   'labels': [],
   'elementId': None,
   'condition': 'Striae distensae',
   'chief_complaint': 'Thin streak-like marks on the trunk',
   'course_trend': 'Stable',
   'comorbidities': 'Multiple mononeuropathy, recurrent inflammatory episodes',
   'plan_followup': 'Monitor for new skin changes and reassess if lesions spread',
   'medication_statement': 'Emollient cream topically BID',
   'notes': 'Skin findings unchanged since onset',
   'encounter': {'id': 'E012',
    'period_start': '2026-03-03',
    'reason_code': 'New skin streaks o

### Coverage Tool (using the ICD-HPO mappings)

In [48]:
graph_client.query("""
    CALL apoc.dv.query('encounter', {patientId: 'P003'}) YIELD node AS v
    WITH apoc.convert.fromJsonList(
           coalesce(
               apoc.any.property(v, 'ICD10_Codes'),
               apoc.any.property(v, '\uFEFFICD10_Codes')
           )
         ) AS codes
    RETURN codes
    LIMIT 1
""")

[{'codes': ['G57.1',
   'T28.1',
   'S84',
   'R52',
   'G58.7',
   'S94.7',
   'R26.2',
   'M54.8',
   'R26.0',
   'G58.7']}]

In [49]:
graph_client.query("""
    CALL apoc.dv.query('encounter', {patientId: 'P003'}) 
    YIELD node AS v

    WITH
    apoc.convert.fromJsonList(
        coalesce(
        apoc.any.property(v, 'ICD10_Codes'),
        apoc.any.property(v, '\uFEFFICD10_Codes')
        )
    ) AS codes,
    v AS _Patient
    WITH codes, _Patient
    LIMIT 1

    UNWIND codes AS code

    MATCH (d:IcdDisease {id: code})
        <-[:UMLS_TO_ICD]-
        (u:Umls)
        -[:UMLS_TO_HPO_PHENOTYPE]->
        (h:HpoPhenotype)

    RETURN DISTINCT
    apoc.any.property(_Patient, 'patientId') AS PatientId,
    h.label AS HPO_Label,
    h.id AS hpo_id,
    'Patient[' + apoc.any.property(_Patient, 'patientId') + ']'
    + ' --ICD10[' + code + ']--> '
    + 'HPO[' + h.id + ' | ' + h.label + ']' AS PathDescription;

""")


[{'PatientId': 'P003',
  'HPO_Label': 'Pain',
  'hpo_id': 'HP:0012531',
  'PathDescription': 'Patient[P003] --ICD10[R52]--> HPO[HP:0012531 | Pain]'},
 {'PatientId': 'P003',
  'HPO_Label': 'Multiple mononeuropathy',
  'hpo_id': 'HP:0032018',
  'PathDescription': 'Patient[P003] --ICD10[G58.7]--> HPO[HP:0032018 | Multiple mononeuropathy]'},
 {'PatientId': 'P003',
  'HPO_Label': 'Gait ataxia',
  'hpo_id': 'HP:0002066',
  'PathDescription': 'Patient[P003] --ICD10[R26.0]--> HPO[HP:0002066 | Gait ataxia]'}]

In [50]:
coverage_payload = {
    "patient_id": "P003",
    "limit": 20
}

coverage_tool_output = build_patient_coverage_tool(chat_client).invoke(coverage_payload)
coverage_tool_output


/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=PatientCoverageResponse(c...ny missing HPO terms.']), input_type=PatientCoverageResponse])
  return self.__pydantic_serializer__.to_python(
Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=2, column=5, offset=5>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 5, 'line': 2, 'column': 5}, 'OPERATI

{'cypher': 'MULTI-STEP PIPELINE: get_patient_icd_codes -> map_icd_to_hpo -> rollup_hpo_to_ancestors -> compute_coverage',
 'rows': [{'diseaseId': 'ORPHA:117',
   'diseaseName': 'Behçet disease',
   'covered': 7,
   'total': 85,
   'coveragePct': 8.2,
   'missingHpoIds': ['HP:0002204',
    'HP:0002202',
    'HP:0001954',
    'HP:0007813',
    'HP:0200039',
    'HP:0001637',
    'HP:0000708',
    'HP:0100820',
    'HP:0012649',
    'HP:0002376',
    'HP:0001287',
    'HP:0001482',
    'HP:0000031',
    'HP:0000155',
    'HP:0002014',
    'HP:0001289',
    'HP:0001369',
    'HP:0000518',
    'HP:0100584',
    'HP:0100614',
    'HP:0100653',
    'HP:0200034',
    'HP:0100654',
    'HP:0000083',
    'HP:0002383',
    'HP:0001250',
    'HP:0012219',
    'HP:0002024',
    'HP:0001097',
    'HP:0001251',
    'HP:0012819',
    'HP:0001653',
    'HP:0002027',
    'HP:0000488',
    'HP:0002105',
    'HP:0002102',
    'HP:0001658',
    'HP:0001733',
    'HP:0006824',
    'HP:0001659',
    'HP:0002

# GraphRAG Agent

In [51]:
from llm.agent import run_agent


In [53]:
out = run_agent("Find phenotypes associated with the influenza disease and include the mapping support text.")
print(out["final_answer"])

/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GuardrailsDecision(decisi...the scope of this app.'), input_type=GuardrailsDecision])
  return self.__pydantic_serializer__.to_python(
/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ValidateCypherOutput(errors=[]), input_type=ValidateCypherOutput])
  return self.__pydantic_serializer__.to_python(


The following phenotypes are associated with influenza disease:

*   Acute bronchitis
*   Infection due to encapsulated bacteria
*   Recurrent Haemophilus influenzae infections
*   Severe influenza infection
*   Triggered by vaccination
*   Unusual viral infection


In [54]:
out = run_agent("Provide me details on the follow up plan of patient P003 in the latest encounter.")
print(out["final_answer"])

/Users/giuseppefutia/Desktop/code/graph-med/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GuardrailsDecision(decisi...f a domain gatekeeper.'), input_type=GuardrailsDecision])
  return self.__pydantic_serializer__.to_python(


Answer:
The follow-up plan for patient P003 in the latest encounter on 2026-03-03 is to monitor for new skin changes and reassess if the lesions spread. The patient is instructed to use regular emollients.



In [60]:
out = run_agent("Show possible diseases by HPO coverage for patientId:'P003'. Report the covered, total, and percentage coverage.")
print(out["final_answer"])

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=2, column=5, offset=5>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 5, 'line': 2, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    CALL {\n        UNWIND $codes AS code\n        MATCH (:IcdDisease {id: code})-[:ICD_MAPS_TO_HPO_BY_EMBEDDING]->(h:HpoPhenotype)\n        RETURN DISTINCT h.id AS hpo_id\n        \n        UNION\n        \n        UNWIND $codes AS code\n        MATCH (:IcdDisease {id: code})<-[:UMLS_TO_ICD]-(:Umls)-[:UMLS_TO_HPO_PHENOTYPE]->(h:HpoPhenotype)\

DEBUG: Retrieved ICD codes: ['A68', 'A68.9', 'B48.8', 'F40.0', 'F43.8', 'F51.4', 'F59', 'F90.0', 'G02.1', 'G03.0', 'G44', 'G44.2', 'G57.1', 'G58.7', 'H30.0', 'H30.2', 'H35.0', 'H43.0', 'H49.1', 'H49.2', 'H49.3', 'H50.2', 'H53.2', 'H53.9', 'H57.1', 'I44.0', 'I47', 'I48.4', 'I49.4', 'K03.0', 'K07.2', 'K07.6', 'L90.6', 'M13.1', 'M23.4', 'M24.5', 'M25.4', 'M25.5', 'M25.6', 'M54.2', 'M54.8', 'R00.2', 'R11', 'R22.4', 'R23.8', 'R25.0', 'R26.0', 'R26.2', 'R41.1', 'R41.8', 'R42', 'R50', 'R50.8', 'R51', 'R52', 'R52.9', 'R53', 'R55', 'R70.0', 'R83.4', 'S03.4', 'S13.4', 'S84', 'S94.7', 'T03.0', 'T28.1', 'W10', 'Z50']
Based on the provided data, the following diseases are covered for patientId 'P003':

*   Developmental delay, impaired speech, and behavioral abnormalities (OMIM:619475) - 18/188 (9.6%)
*   Behçet disease (ORPHA:117) - 17/85 (20.0%)
*   Brucellosis (ORPHA:1304) - 16/77 (20.8%)
*   Lyme disease (ORPHA:900) - 16/24 (66.7%)
*   Giant cell arteritis (ORPHA:289390) - 15/69 (21.7%)
*   Afr

In [61]:
raw_response = chat_client.invoke(
    "A patient presents with the following ICD-10 codes from multiple encounters: "
    "G57.1 (Meralgia paraesthetica), G58.7 (Mononeuritis multiplex), "
    "G02.1, G03.0, G44, R26.0 (Ataxic gait), R26.2, R52 (Pain), M54.2 (Cervicalgia), "
    "R42 (Dizziness), R50 (Fever), R53 (Malaise and fatigue), R55 (Syncope). "
    "Based on HPO phenotype coverage, what diseases could explain this clinical profile? "
    "Report possible diseases with estimated phenotype coverage."
)
print(raw_response.content)

Okay, let's break down the patient's presentation and use HPO phenotype coverage to suggest possible diseases.

**Understanding the Patient's Presentation**

The patient has a constellation of symptoms including:

*   **Numbness/Pain (Meralgia paraesthetica):** G57.1
*   **Neuropathy (Mononeuritis multiplex):** G58.7
*   **Gait Disturbance (Ataxic gait):** R26.0, R26.2
*   **Pain:** R52
*   **Neck Pain (Cervicalgia):** M54.2
*   **Dizziness:** R42
*   **Fever:** R50
*   **Fatigue/Malaise:** R53, R55 (Syncope)

This suggests a systemic or widespread process affecting the nerves, potentially with some localized involvement. The presence of fever and malaise points towards an inflammatory or infectious component.

**HPO Phenotype Coverage and Possible Diseases**

Here's a breakdown of potential diseases, ranked by likelihood based on the presented symptoms, along with estimated HPO phenotype coverage:

1.  **Systemic Lupus Erythematosus (SLE) (HP:0000256)**

    *   **HPO Coverage:**  SLE

In [ ]:
from llm.query_factory import get_patient_icd_codes, map_icd_to_hpo, rollup_hpo_to_ancestors, compute_coverage

# Step 1: Pipeline — get patient phenotypes and coverage
icd_codes = get_patient_icd_codes("P003")
hpo_ids = map_icd_to_hpo(icd_codes)
ancestors = rollup_hpo_to_ancestors(hpo_ids)
coverage = compute_coverage(ancestors, limit=10)

# Resolve HPO IDs to labels
hpo_labels = graph_client.query("""
    UNWIND $ids AS hid
    MATCH (h:HpoPhenotype {id: hid})
    RETURN h.label AS label
    ORDER BY label
""", {"ids": hpo_ids})

symptom_list = "\n".join(f"- {r['label']}" for r in hpo_labels)

print(f"Patient P003: {len(icd_codes)} ICD codes → {len(hpo_ids)} HPO phenotypes → {len(ancestors)} ancestors")
print(f"\nHPO phenotypes:\n{symptom_list}")

# Step 2: Pure LLM — same symptoms, no graph candidates
print("\n" + "=" * 80)
print("LLM-only differential diagnosis (no graph context)")
print("=" * 80)

raw_response = chat_client.invoke(
    "A patient presents with the following phenotypes (HPO terms):\n"
    f"{symptom_list}\n\n"
    "What are the most probable 10 diseases that could explain this clinical profile? "
)
print(raw_response.content)

# Step 3: Graph pipeline coverage
print("\n" + "=" * 80)
print("Graph pipeline coverage (HPO disease matching)")
print("=" * 80)

for row in coverage:
    print(f"  {row['diseaseName']}: {row['covered']}/{row['total']} ({row['coveragePct']}%)")


Patient P003: 68 ICD codes → 73 HPO phenotypes → 226 ancestors

HPO phenotypes:
- Abducens palsy
- Abnormal CSF/serum albumin ratio
- Abnormal head movements
- Abnormal retinal vascular morphology
- Abnormality of mental function
- Abnormality of the dentition
- Abnormality of the integument
- Agoraphobia
- Amyotrophy of ankle musculature
- Anterograde memory impairment
- Arthralgia
- Atrial flutter
- Atrophic scars
- Back pain
- Cervical subluxation
- Chorioretinitis
- Chronic pain
- Diagnostic behavioral phenotype
- Diplopia
- Distal lower limb muscle weakness
- Elevated erythrocyte sedimentation rate
- Erythromelalgia
- Esophageal ulceration
- Excessive dental attrition
- Falls
- Fatigue
- Fever
- First degree atrioventricular block
- Fourth cranial nerve palsy
- Fungal meningitis
- Gait ataxia
- Gait disturbance
- Headache
- Hyperactivity
- Hypertropia
- Intense psychological distress to cues
- Invasive dermatophyte infection
- Jaw pain
- Joint contracture
- Joint stiffness
- Knee 

# Embedding API

In [ ]:
cfg = load_config_api("embedding", path="../config.ini")
api = ApiClient(cfg)

In [ ]:
body, status, headers = api.post('/embed', {'input': ['Who are you?', 'What is your name?']})
print("Status:", status)
print("Headers:", headers)
print("Body:", body)

# Semantic Search

In [ ]:
import configparser
from database.neo4j_db import Neo4jGraphDB

config = configparser.ConfigParser()
config.read('../config.ini')
neo4j_graph = Neo4jGraphDB(database=config["neo4j"]["database"])
driver = neo4j_graph._driver
with driver.session(database=neo4j_graph._database) as session:
    result = session.run("MATCH (n) RETURN count(n) AS node_count")
    node_count = result.single()["node_count"]
print(f"Total number of nodes in the Neo4j database: {node_count}")


In [ ]:
user_query = "Muscular dystrophy."
q_embed = api.post('/embed', {'input': [user_query]})[0]['data'][0]
k = 10

with driver.session(database=neo4j_graph._database) as sess:
    result = sess.run("""
        CALL db.index.vector.queryNodes('hpo_phenotype_embedding', $k, $qe)
        YIELD node, score
        RETURN node.id as id, node.label as label, score
        ORDER BY score DESC
        LIMIT $k
    """, k=k, qe=q_embed)
    for rec in result:
        print(f"{rec['score']:.3f}  {rec['label']}  (id={rec['id']})")